In [1]:
!pip install -q transformers accelerate bitsandbytes
!pip install -q transformers accelerate bitsandbytes datasets --upgrade
!pip install -q huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 34.4 MB/s eta 0:00:00:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 105.7 MB/s eta 0:00:0000:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.3/512.3 kB 30.1 MB/s eta 0:00:00


## Imports

In [2]:
import sys
import os
import json
import pickle
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from tqdm.auto import tqdm
import re
import warnings
from google.colab import drive

warnings.filterwarnings('ignore')

print(f"✓ PyTorch: {torch.__version__}")
print(f"✓ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")

✓ PyTorch: 2.8.0+cu126
✓ CUDA available: True
✓ GPU: Tesla T4


## Load DialogRE Train Set

In [3]:
DATA_PATH = '/kaggle/input/lama3-data'
TRAIN_PKL = os.path.join(DATA_PATH, 'train_split.pkl')


print(f"Looking for data in: {DATA_PATH}")

train_data = None

if os.path.exists(TRAIN_PKL):
    print(f"Found train.pkl, loading...")
    with open(TRAIN_PKL, 'rb') as f:
        train_data = pickle.load(f)
    print(f"✓ Loaded train.pkl: {len(train_data)} dialogues")
else:
    print(f" Error: Could not find train.json or train.pkl in {DATA_PATH}")
    print("Please upload the DialogRE dataset!")
    sys.exit(1)

# Show sample (Adjusted to reflect train_data as a list of dictionaries)
print("\n📋 Data structure:")
print(f"Total pre-processed relations: {len(train_data)}")
print("\nSample relation structure:")

if train_data:
    sample = train_data[0]
    print(f"  - Keys available: {list(sample.keys())}")
    print(f"  - Sample conversation_id: {sample.get('conversation_id', 'N/A')}")
    print(f"  - Sample subject: {sample.get('subject', 'N/A')}")
    print(f"  - Sample object: {sample.get('object', 'N/A')}")
    print(f"  - Sample relation: {sample.get('relation', 'N/A')}")
    print(f"  - Sample processed_text: {sample.get('processed_text', 'N/A')[:100]}...")
else:
    print("  No data loaded to show a sample.")

Looking for data in: /kaggle/input/lama3-data
Found train.pkl, loading...
✓ Loaded train.pkl: 50831 dialogues

📋 Data structure:
Total pre-processed relations: 50831

Sample relation structure:
  - Keys available: ['conversation_id', 'dataset', 'subject', 'object', 'relation', 'is_positive', 'processed_text', 'input_ids', 'attention_mask', 'token_type_ids', 'entity_positions', 'label_id']
  - Sample conversation_id: conv_322
  - Sample subject: baby
  - Sample object: monkey
  - Sample relation: no_relation
  - Sample processed_text: [Turn 27] [SPEAKER] Speaker 1: Are you kidding? Okay, look. I-I studied evolution. Remember, evoluti...


## Relation Extraction

In [4]:
def extract_relations_from_dialogre(train_data, max_samples=1200):
    """
    Extract relation triples from the pre-processed DialogRE dataset.
    Each entry in train_data is expected to be a dictionary representing a single relation.
    The 'processed_text' field is used as the sentence for analysis.
    """

    # Relations that indicate emotional/social connections
    SENTIMENT_RELATIONS = {'per:friends', 'per:siblings', 'per:negative_impression',
                           'per:girl/boyfriend', 'per:positive_impression',
                           'per:spouse', 'per:pet', 'per:children', 'per:parents',
                           'per:roommate', 'per:neighbor', 'per:other_family', 'per:boss',
                           'per:subordinate', 'per:origin'}

    relations_extracted = []

    for dialogue_idx, relation_data_dict in enumerate(tqdm(train_data, desc="Extracting relations")):
        if len(relations_extracted) >= max_samples:
            break

        subject = relation_data_dict.get('subject', 'N/A')
        obj = relation_data_dict.get('object', 'N/A')
        current_relation_type = relation_data_dict.get('relation', 'unknown')

        if current_relation_type not in SENTIMENT_RELATIONS:
            continue

        sentence = relation_data_dict.get('processed_text', '')

        # Calculate context length (total characters)
        context_length = len(sentence)

        conv_id = relation_data_dict.get('conversation_id', str(dialogue_idx))

        # Add to extracted relations
        relations_extracted.append({
            'conv_id': conv_id,
            'sentence': sentence,
            'subject': subject,
            'object': obj,
            'relation': current_relation_type,
            'subject_name': subject, 
            'object_name': obj,    
            'context_length': context_length,
        })

    return pd.DataFrame(relations_extracted)

## Load Data for Few-shot

In [5]:
# Option 1: Load from your manual annotations (RECOMMENDED)
FEW_SHOT_FILE = '/kaggle/input/lama3-data/annotation_samples_150.csv'

# Try to load few-shot examples
few_shot_examples = []

if os.path.exists(FEW_SHOT_FILE):
    print(f"Loading few-shot examples from: {FEW_SHOT_FILE}")
    df_manual = pd.read_csv(FEW_SHOT_FILE)

    # Convert to few-shot format
    # We take a subset to ensure variety; usually 3-5 examples in the actual prompt is best
    for _, row in df_manual.iterrows():
        sentence = row.get('sentence', '')
        # Using 'manual_label' as per your CSV structure
        label = str(row.get('manual_label', 'neutral')).lower()

        # Create Domain-Specific Reasoning for Social Graph Mining
        # This provides the "Chain of Thought" the LLM needs to mimic
        if label == 'positive':
            reasoning = "The speaker uses language of inclusion, gratitude, or praise, indicating a favorable social bond and positive emotional attribution toward the target."
        elif label == 'negative':
            reasoning = "The interaction contains elements of conflict, social rejection, or hostile intent, establishing a negative emotional attribution between the parties."
        else:
            reasoning = "The dialogue is task-oriented, logistical, or observational. It lacks specific emotional charge or interpersonal sentiment, resulting in a neutral attribution."

        few_shot_examples.append({
            'sentence': sentence,
            'label': label,
            'reasoning': reasoning,
            'conv_id': row.get('conv_id', 0) # Keeping ID for reference
        })

    print(f"✓ Loaded {len(few_shot_examples)} manual annotations as potential few-shot examples")

print(f"\n✓ Few-shot examples ready: {len(few_shot_examples)}")

Loading few-shot examples from: /kaggle/input/lama3-data/annotation_samples_150.csv
✓ Loaded 150 manual annotations as potential few-shot examples

✓ Few-shot examples ready: 150


In [6]:
from huggingface_hub import login
login()

## Load Llama3 Moodel

In [8]:
# Quantization config (4-bit for memory efficiency)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Model name
MODEL_NAME = "meta-llama/Meta-Llama-3-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3-8B-Instruct")
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'  # Fix the warning
    
    # Load model
model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Meta-Llama-3-8B-Instruct",
    device_map="auto",
    torch_dtype=torch.float16
    )

print("\n✓ Llama-3-8B-Instruct loaded successfully")
print(f"✓ Model device: {model.device}")

tokenizer_config.json:   0%|          | 0.00/51.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!
2025-12-24 11:16:56.276586: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766575016.501602      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766575016.573728      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1766575017.149945      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766575017.149973      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766575017.149976      55

...

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]


✓ Llama-3-8B-Instruct loaded successfully
✓ Model device: cuda:0


##  Create Few-Shot Prompt Function

In [9]:
def create_few_shot_prompt(sentence, few_shot_examples, num_examples=10):
    """
    Create a few-shot prompt for Llama-3.
    Uses random selection from examples to provide variety.
    """

    # Randomly select examples (but keep consistent across calls with same sentence)
    np.random.seed(hash(sentence) % 10000)
    selected_examples_indices = np.random.choice(
        range(len(few_shot_examples)),
        size=min(num_examples, len(few_shot_examples)),
        replace=False
    )
    selected_few_shot_examples = [few_shot_examples[i] for i in selected_examples_indices]

    prompt = "You are an AI expert in Social Graph Mining. Classify the emotional attribution between the Subject and Object in the following dialogue.\n\n"

    # The loop for adding few-shot examples
    for i, example in enumerate(selected_few_shot_examples, 1):
        # Use the reasoning already stored in the example
        reasoning_text = example['reasoning']

        prompt += f'''Example {i}:
Sentence: "{example['sentence']}"
Reasoning: {reasoning_text}
Classification: {example['label'].lower()}

'''

    # Add the target query
    prompt += f'''Now classify this sentence:
Sentence: "{sentence}"

Provide your answer in EXACTLY this format:
Reasoning: [your step-by-step reasoning]
Classification: [positive/negative/neutral]
Confidence: [a number between 0.0 and 1.0]

Your answer:'''

    return prompt

## Prediction Function (Batch)

In [10]:
def predict_with_llama3_batch(sentences, few_shot_examples, max_retries=2):
    """
    Predict sentiment using Llama-3 with few-shot prompting, processing a batch of sentences.
    """

    results = []

    # Create prompts for all sentences in the batch
    prompts = [create_few_shot_prompt(sentence, few_shot_examples) for sentence in sentences]

    # Tokenize batch
    inputs = tokenizer(prompts, return_tensors="pt", truncation=True, max_length=2048, padding=True)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    # Generate
    try:
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=150,
                temperature=0.1,
                do_sample=True,
                top_p=0.9,
                pad_token_id=tokenizer.eos_token_id
            )

        # Decode all outputs and parse results for each sentence
        # The generate function returns outputs including the input prompts for each batch item
        decoded_responses = tokenizer.batch_decode(outputs[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)

        for response_text in decoded_responses:
            # Extract answer part
            answer_start = response_text.find("Your answer:")
            if answer_start != -1:
                answer = response_text[answer_start + len("Your answer:"):].strip()
            else:
                answer = response_text.strip() # Fallback if 'Your answer:' is missing, take whole response

            answer_lower = answer.lower()

            # Parse label
            label = 'neutral'
            if 'classification: positive' in answer_lower or '\npositive' in answer_lower:
                label = 'positive'
            elif 'classification: negative' in answer_lower or '\nnegative' in answer_lower:
                label = 'negative'
            elif 'classification: neutral' in answer_lower or '\nneutral' in answer_lower:
                label = 'neutral'
            else:
                # Fallback: look for keywords
                # Use re.search to find label more flexibly and split the string
                label_match = re.search(r'(positive|negative|neutral)', answer_lower)
                if label_match:
                    label = label_match.group(1)
                else:
                    label = 'neutral'

            # Parse confidence
            confidence = 0.7
            conf_match = re.search(r'confidence:?\s*([0-9.]+)', answer_lower)
            if conf_match:
                try:
                    confidence = float(conf_match.group(1))
                    if confidence > 1.0:
                        confidence = confidence / 10.0
                    confidence = max(0.0, min(1.0, confidence))
                except:
                    confidence = 0.7

            # Parse reasoning more robustly
            reasoning = ""
            # First, try to find the explicit "Reasoning:" prefix
            explicit_reasoning_match = re.search(r'reasoning:\s*(.+?)(?=classification:|confidence:|$)',
                                                  answer_lower, re.DOTALL)
            if explicit_reasoning_match:
                reasoning = explicit_reasoning_match.group(1).strip()
            else:
                # If explicit "Reasoning:" is not found, try to infer it.
                # Assume reasoning is the text before the classification or confidence keywords.
                temp_text = answer_lower

                # Remove the classification part if found
                classification_idx = temp_text.find('classification:')
                if classification_idx != -1:
                    temp_text = temp_text[:classification_idx]

                # Remove the confidence part if found (after classification, or if classification was not there)
                confidence_idx = temp_text.find('confidence:')
                if confidence_idx != -1:
                    temp_text = temp_text[:confidence_idx]

                # The remaining `temp_text` is a candidate for reasoning.
                # We need to remove the label if it's at the very beginning of this candidate.
                # Example: "positive\nSome reasoning..." -> remove "positive\n"
                cleaned_reasoning_candidate = temp_text.strip()

                # Check if the candidate starts with the extracted label (case-insensitive and whole word)
                label_pattern = r'^' + re.escape(label) + r'\b' # ensure it matches whole word label at start
                if re.match(label_pattern, cleaned_reasoning_candidate, re.IGNORECASE):
                    potential_reasoning_after_label = re.sub(label_pattern, '', cleaned_reasoning_candidate, 1, flags=re.IGNORECASE).strip()
                    if potential_reasoning_after_label:
                        reasoning = potential_reasoning_after_label
                    else: # If only label, no further text, assume no reasoning
                        reasoning = ""
                else:
                    reasoning = cleaned_reasoning_candidate

            # Limit reasoning length
            reasoning = reasoning[:200].strip()

            # Clean up any residual prompt instructions from reasoning if it's too short
            if len(reasoning) > 0 and len(reasoning) < 50 and \
               ("please provide your answer" in reasoning or "your step-by-step reasoning" in reasoning or \
                "the confidence level is subjective" in reasoning):
                reasoning = ""

            results.append({
                'label': label,
                'confidence': confidence,
                'reasoning': reasoning,
                'raw_response': answer[:500] # Limit raw response length
            })

    except Exception as e:
        print(f"Error in prediction: {e}")
        for _ in sentences:
            results.append({
                'label': 'neutral',
                'confidence': 0.0,
                'reasoning': f"Error: {str(e)}",
                'raw_response': ''
            })

    return results

## Test Predictions (Batch)

In [11]:
print("\n" + "="*70)
print("STEP 5: Testing Llama-3 Batch Predictions")
print("="*70)

test_sentences = [
    "Bob really admires Mary's intelligence",
    "Mary hates Jane with passion",
    "Jane adores and loves Bob very much"
]

# Use the new batch prediction function
results_batch = predict_with_llama3_batch(test_sentences, few_shot_examples)

for i, sent in enumerate(test_sentences):
    result = results_batch[i]
    print(f"\n{'='*60}")
    print(f"Sentence: {sent}")
    print(f"Label: {result['label']}")
    print(f"Confidence: {result['confidence']:.3f}")
    print(f"Reasoning: {result['reasoning']}")

print("\n✓ Test predictions complete!")


STEP 5: Testing Llama-3 Batch Predictions

Sentence: Bob really admires Mary's intelligence
Label: positive
Confidence: 0.900
Reasoning: the sentence uses language of admiration and appreciation, indicating a positive emotional attribution toward mary.

Sentence: Mary hates Jane with passion
Label: negative
Confidence: 1.000
Reasoning: the sentence explicitly states a strong negative emotion (hate) towards jane, indicating a negative emotional attribution between mary and jane.

Sentence: Jane adores and loves Bob very much
Label: positive
Confidence: 0.950
Reasoning: the sentence explicitly states that jane has strong positive emotions towards bob, using the words "adores" and "loves". this indicates a strong social bond and positive emotional attribution.

✓ Test predictions complete!


## CELL 9: Label All Samples

In [12]:
# Call the function to extract relations and create df_relations
print("Extracting relations from train_data...")
df_relations = extract_relations_from_dialogre(train_data, max_samples=1500) # You can adjust max_samples
print(f"✓ Extracted {len(df_relations)} relations.")


print("\n" + "="*70)
print("STEP 6: Labeling All Samples with Llama-3")
print("="*70)
print(f"Total samples to label: {len(df_relations)}")

BATCH_SIZE = 4  

# Estimate time based on batch size
estimated_total_batches = len(df_relations) / BATCH_SIZE
print(f"Estimated time: ~{estimated_total_batches * 4 / 60:.0f} minutes (assuming 4 seconds per batch of {BATCH_SIZE})")
print("Progress will be saved every 200 samples.\n")

# Label all samples in batches
predictions = []
checkpoint_interval = 200

# Prepare lists for batch processing
batch_sentences = []
batch_rows = []

for idx, row in tqdm(df_relations.iterrows(), total=len(df_relations), desc="Labeling"):
    batch_sentences.append(row['sentence'])
    batch_rows.append(row)

    if len(batch_sentences) == BATCH_SIZE or idx == len(df_relations) - 1:
        # Predict for the current batch
        batch_results = predict_with_llama3_batch(batch_sentences, few_shot_examples)

        # Store results for each item in the batch
        for i, result in enumerate(batch_results):
            original_row = batch_rows[i]
            predictions.append({
                'conv_id': original_row['conv_id'],
                'subject': original_row['subject'],
                'object': original_row['object'],
                'sentence': original_row['sentence'],
                'llm_label': result['label'],
                'llm_confidence': result['confidence'],
                'llm_reasoning': result['reasoning']
            })

        # Clear batch lists
        batch_sentences = []
        batch_rows = []

        # Checkpoint save (based on total samples processed so far)
        if len(predictions) % checkpoint_interval == 0:
            df_checkpoint = pd.DataFrame(predictions)
            df_checkpoint.to_csv(f'checkpoint_{len(predictions)}.csv', index=False)
            print(f"\n✓ Checkpoint: {len(predictions)}/{len(df_relations)} samples")

# Create final dataframe
df_labeled = pd.DataFrame(predictions)

print(f"\n✓ Labeling complete!")
print(f"Total labeled: {len(df_labeled)}")

# Save all labels
df_labeled.to_csv('llm_labels_all.csv', index=False)
print("\n✓ Saved: llm_labels_all.csv")

Extracting relations from train_data...


Extracting relations:   0%|          | 0/50831 [00:00<?, ?it/s]

✓ Extracted 1500 relations.

STEP 6: Labeling All Samples with Llama-3
Total samples to label: 1500
Estimated time: ~25 minutes (assuming 4 seconds per batch of 4)
Progress will be saved every 200 samples.



Labeling:   0%|          | 0/1500 [00:00<?, ?it/s]


✓ Checkpoint: 200/1500 samples

✓ Checkpoint: 400/1500 samples

✓ Checkpoint: 600/1500 samples

✓ Checkpoint: 800/1500 samples

✓ Checkpoint: 1000/1500 samples

✓ Checkpoint: 1200/1500 samples

✓ Checkpoint: 1400/1500 samples

✓ Labeling complete!
Total labeled: 1500

✓ Saved: llm_labels_all.csv


## Quality Control

In [13]:
print("\n" + "="*70)
print("STEP 7: Quality Control")
print("="*70)

# Distribution before filtering
print("\nLabel distribution (before filtering):")
print(df_labeled['llm_label'].value_counts())

print("\nConfidence distribution:")
print(df_labeled['llm_confidence'].describe())

# Filter by confidence
CONFIDENCE_THRESHOLD = 0.7

df_filtered = df_labeled[df_labeled['llm_confidence'] >= CONFIDENCE_THRESHOLD].copy()

print(f"\n✓ Filtered by confidence >= {CONFIDENCE_THRESHOLD}")
print(f"  Before: {len(df_labeled)} samples")
print(f"  After: {len(df_filtered)} samples")
print(f"  Kept: {len(df_filtered)/len(df_labeled)*100:.1f}%")

print("\nLabel distribution (after filtering):")
print(df_filtered['llm_label'].value_counts())

# Save filtered
df_filtered.to_csv('llm_labels_filtered.csv', index=False)
print("\n✓ Saved: llm_labels_filtered.csv")


STEP 7: Quality Control

Label distribution (before filtering):
llm_label
positive    737
neutral     391
negative    372
Name: count, dtype: int64

Confidence distribution:
count    1500.000000
mean        0.855167
std         0.065201
min         0.500000
25%         0.800000
50%         0.900000
75%         0.900000
max         0.950000
Name: llm_confidence, dtype: float64

✓ Filtered by confidence >= 0.7
  Before: 1500 samples
  After: 1485 samples
  Kept: 99.0%

Label distribution (after filtering):
llm_label
positive    737
neutral     377
negative    371
Name: count, dtype: int64

✓ Saved: llm_labels_filtered.csv


## Balance Classes 

In [14]:
print("\n" + "="*70)
print("STEP 8: Balancing Classes")
print("="*70)

from sklearn.utils import resample

# Separate by class
df_pos = df_filtered[df_filtered['llm_label'] == 'positive']
df_neg = df_filtered[df_filtered['llm_label'] == 'negative']
df_neu = df_filtered[df_filtered['llm_label'] == 'neutral']

print(f"\nClass sizes before balancing:")
print(f"  Positive: {len(df_pos)}")
print(f"  Negative: {len(df_neg)}")
print(f"  Neutral: {len(df_neu)}")

# Target size
min_size = min(len(df_pos), len(df_neg), len(df_neu))
TARGET_SIZE = max(min_size, 400)  # At least 400 per class

print(f"\nTarget size per class: {TARGET_SIZE}")

# Resample
if len(df_pos) >= TARGET_SIZE:
    df_pos_balanced = resample(df_pos, n_samples=TARGET_SIZE, random_state=42)
else:
    df_pos_balanced = df_pos

if len(df_neg) >= TARGET_SIZE:
    df_neg_balanced = resample(df_neg, n_samples=TARGET_SIZE, random_state=42)
else:
    df_neg_balanced = df_neg

if len(df_neu) >= TARGET_SIZE:
    df_neu_balanced = resample(df_neu, n_samples=TARGET_SIZE, random_state=42)
else:
    df_neu_balanced = df_neu

# Combine
df_balanced = pd.concat([df_pos_balanced, df_neg_balanced, df_neu_balanced])
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\nClass sizes after balancing:")
print(df_balanced['llm_label'].value_counts())

print(f"\n✓ Final balanced dataset: {len(df_balanced)} samples")

# Save
df_balanced.to_csv('llm_labeled_balanced_final.csv', index=False)
print("\n✓ Saved: llm_labeled_balanced_final.csv")


STEP 8: Balancing Classes

Class sizes before balancing:
  Positive: 737
  Negative: 371
  Neutral: 377

Target size per class: 400

Class sizes after balancing:
llm_label
positive    400
neutral     377
negative    371
Name: count, dtype: int64

✓ Final balanced dataset: 1148 samples

✓ Saved: llm_labeled_balanced_final.csv


## Final Summary 

In [ ]:
print("\n" + "="*70)
print("FINAL SUMMARY")
print("="*70)

print(f"\n Dataset Statistics:")
print(f"  Original relations: {len(df_relations)}")
print(f"  After confidence filter: {len(df_filtered)}")
print(f"  Final balanced: {len(df_balanced)}")

print(f"\n📁 Files Created:")
print(f"  1. dialogre_relations_extracted.csv")
print(f"  2. llm_labels_all.csv")
print(f"  3. llm_labels_filtered.csv")
print(f"  4. llm_labeled_balanced_final.csv")

print("\n DONE! Download 'llm_labeled_balanced_final.csv'")

print("\n" + "="*70)
print("NEXT STEPS:")
print("="*70)
print("1. Download llm_labeled_balanced_final.csv")
print("2. Combine with your 150 manual annotations")
print("3. Fine-tune RoBERTa on combined dataset")
print("4. Expected improvement: F1 0.55 → 0.75-0.80")
print("="*70)

# Display samples
print("\n Sample of final dataset:")
print(df_balanced[['subject', 'object', 'sentence', 'llm_label', 'llm_confidence']].head(10).to_string())
